# Explore the Munich CKAN API

This notebook explores the Open Data Portal Munich CKAN API and collects the information needed for `smolnalysis` training-data generation.

Goals:

- inspect the CKAN API surface
- search packages/datasets
- inspect dataset metadata and resources
- sample records from datastore-backed resources
- build a small catalog snapshot for later training-data generation

Munich portal API help: https://opendata.muenchen.de/pages/hilfe

In [25]:
from __future__ import annotations

import json
import time
import xml.etree.ElementTree as ET
from html.parser import HTMLParser
from pathlib import Path
from typing import Any
from urllib.parse import urlencode
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

BASE_URL = "https://opendata.muenchen.de/api/3/action"
USER_AGENT = "smolnalysis-ckan-explorer/0.1"

RAW_DATA_DIR = Path("../training/data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL

'https://opendata.muenchen.de/api/3/action'

## CKAN helper

CKAN actions are exposed as JSON endpoints under `/api/3/action/{action_name}`. This helper keeps calls consistent and raises useful errors when the API returns `success: false`.

In [26]:
def ckan_action(action: str, params: dict[str, Any] | None = None, sleep_s: float = 0.0) -> Any:
    """Call a CKAN action endpoint and return the `result` payload."""
    params = params or {}
    query = f"?{urlencode(params, doseq=True)}" if params else ""
    url = f"{BASE_URL}/{action}{query}"
    request = Request(url, headers={"User-Agent": USER_AGENT})

    try:
        with urlopen(request, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        error_body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError({"url": url, "status": exc.code, "body": error_body[:2000]}) from exc

    if not payload.get("success"):
        raise RuntimeError({"action": action, "params": params, "error": payload.get("error")})

    if sleep_s:
        time.sleep(sleep_s)

    return payload["result"]


def compact_package(package: dict[str, Any]) -> dict[str, Any]:
    """Keep the fields that are useful for search, routing, and training-data generation."""
    return {
        "id": package.get("id"),
        "name": package.get("name"),
        "title": package.get("title"),
        "notes": package.get("notes"),
        "tags": [tag.get("name") for tag in package.get("tags", [])],
        "groups": [group.get("name") for group in package.get("groups", [])],
        "organization": (package.get("organization") or {}).get("title"),
        "metadata_created": package.get("metadata_created"),
        "metadata_modified": package.get("metadata_modified"),
        "resources": [compact_resource(resource) for resource in package.get("resources", [])],
    }


def compact_resource(resource: dict[str, Any]) -> dict[str, Any]:
    return {
        "id": resource.get("id"),
        "name": resource.get("name"),
        "description": resource.get("description"),
        "format": resource.get("format"),
        "mimetype": resource.get("mimetype"),
        "url": resource.get("url"),
        "datastore_active": resource.get("datastore_active"),
        "created": resource.get("created"),
        "last_modified": resource.get("last_modified"),
    }

## Site metadata

Start with `site_read` and the global tag/group lists to understand the portal.

In [27]:
site = ckan_action("site_read")
site

True

In [ ]:
tags = ckan_action("tag_list")
groups = ckan_action("group_list")

print(f"tags: {len(tags):,}")
print(f"groups: {len(groups):,}")
print("sample tags:", tags[:30])
print("groups:", groups[:30])

tags: 217
groups: 13
sample tags: ['Abfall', 'Abwasser', 'Abwasserreinigung', 'Adressen', 'Alter', 'Anliegenmanagement', 'application', 'Arbeit', 'Arbeitslosigkeit', 'Arbeitsmarkt', 'Ausleihe', 'Author', 'Auto', 'Barrierefreiheit', 'Bauen', 'Beisetzung', 'Besattungsbezirk', 'Bestand', 'Bestattung', 'Bestattungen', 'Besucherzahlen', 'Bevölkerung', 'Bezirksausschüsse', 'Bibliothek', 'Bier', 'Bierpreis', 'Brücken', 'Bücher', 'Bücherei', 'Bundestagswahl']
groups: ['soci', 'educ', 'ener', 'heal', 'intr', 'just', 'agri', 'gove', 'regi', 'envi', 'tran', 'econ', 'tech']


## Search datasets

`package_search` is the main dataset discovery endpoint. This is the API action the function-calling model will usually call first.

In [29]:
def search_packages(query: str = "", rows: int = 10, start: int = 0, sleep_s: float = 0.0, **extra: Any) -> dict[str, Any]:
    params = {"q": query, "rows": rows, "start": start, **extra}
    return ckan_action("package_search", params, sleep_s=sleep_s)


search_result = search_packages("", rows=10)
print("total packages:", search_result["count"])

packages = search_result["results"]
pd.DataFrame([
    {
        "name": package.get("name"),
        "title": package.get("title"),
        "resources": len(package.get("resources", [])),
        "formats": sorted({resource.get("format") for resource in package.get("resources", []) if resource.get("format")}),
    }
    for package in packages
])

total packages: 336


,name,title,resources,formats
0,solarpotenzial_globalstrahlung_p_02,Solarpotenzialanalyse 2018 der Landeshauptstadt München,4,"[HTML, WMS, XML]"
1,enp_baublock_grundwasserpotential_einteilig_25832,Grundwasserpotential,4,"[HTML, WMS, XML]"
2,abfall_sonstiges_opendata,Sonstige Abfallentsorgungsanlagen (Opendata),7,"[CSV, GeoJSON, HTML, Shape, WMS, XML]"
3,bestand_sbh_opendata,Sozialbürgerhaus,7,"[CSV, GeoJSON, HTML, Shape, WMS, XML]"
4,abfall_gewerbemisch_opendata,Anlagen zur Sortierung von Baustellen- und Gewerbemischabfällen (Opendata),7,"[CSV, GeoJSON, HTML, Shape, WMS, XML]"
5,awm_container_barrierefrei,Barrierefreie Container für Altglas,7,"[CSV, HTML, JSON, WMS, XML]"
6,abfall_betrieb_opendata,Abfallentsorgungsanlagen der Landeshauptstadt München,7,"[CSV, GeoJSON, HTML, Shape, WMS, XML]"
7,vollst_pflegeeinrichtung_opendata,Vollstationäre Pflegeeinrichtungen,7,"[CSV, GeoJSON, HTML, Shape, WMS, XML]"
8,abfall_papier_opendata,Anlagen zur Lagerung und Behandlung von Altpapier (Opendata),7,"[CSV, GeoJSON, HTML, Shape, WMS, XML]"
9,abfall_elektronik_opendata,Anlagen zur Lagerung und Behandlung von Elektronikschrott (Opendata),7,"[CSV, GeoJSON, HTML, Shape, WMS, XML]"


Try realistic user-query keywords. These examples are useful seed material for the tool-calling training set.

In [30]:
queries = [
    "vornamen",
    "bevoelkerung",
    "verkehr",
    "radverkehr",
    "luftqualitaet",
    "schulen",
    "wahl",
    "stadtbezirke",
]

rows = []
for query in queries:
    result = search_packages(query, rows=5, sleep_s=0.1)
    for rank, package in enumerate(result["results"], start=1):
        rows.append({
            "query": query,
            "rank": rank,
            "package_name": package.get("name"),
            "title": package.get("title"),
            "resource_count": len(package.get("resources", [])),
            "formats": sorted({resource.get("format") for resource in package.get("resources", []) if resource.get("format")}),
        })

search_df = pd.DataFrame(rows)
search_df

,query,rank,package_name,title,resource_count,formats
0,vornamen,1,vornamen-von-neugeborenen,Vornamen von Nulljährigen München,13,[CSV]
1,bevoelkerung,1,bevoelkerung,Die Bevölkerung seit 1900,1,[CSV]
2,bevoelkerung,2,bevoelkerung-stadtbezirken,Bevölkerung in den Stadtbezirken,1,[CSV]
3,bevoelkerung,3,bevoelkerung-stadtbezirksteile-muenchen,Bevölkerung Stadtbezirksteile München,1,[CSV]
4,bevoelkerung,4,indikatorenatlas-bevoelkerung-bevoelkerungsdichte-83r65mct,Indikatorenatlas: Bevölkerung - Bevölkerungsdichte,1,[CSV]
5,bevoelkerung,5,indikatorenatlas-bevoelkerung-religionszugehoerigkeit,Indikatorenatlas: Bevölkerung - Religionszugehörigkeit,1,[CSV]
6,verkehr,1,indikatorenatlas-verkehr-motorisierungsgrad-personenkraftwagen-83r65mct,Indikatorenatlas: Verkehr - Motorisierungsgrad Personenkraftwagen,1,[CSV]
7,verkehr,2,indikatorenatlas-verkehr-erstzulassungsanteil-personenkraftwagen-83r65mct,Indikatorenatlas: Verkehr - Erstzulassungsanteil Personenkraftwagen,1,[CSV]
8,verkehr,3,indikatorenatlas-verkehr-motorisierungsgrad-personenkraftwagen-und-kraftraeder-83r65mct,Indikatorenatlas: Verkehr - Motorisierungsgrad Personenkraftwagen und Krafträder,1,[CSV]
9,verkehr,4,haltestellenliste-mvv,Haltestellenliste MVV,1,[CSV]


## Inspect one dataset

`package_show` returns the full dataset metadata, including resource IDs. Pick a dataset from the search results and inspect its resources.

In [31]:
package_name = search_df.iloc[0]["package_name"]
package = ckan_action("package_show", {"id": package_name})

print(package["name"])
print(package["title"])
print((package.get("notes") or "")[:1000])

resource_df = pd.DataFrame(compact_resource(resource) for resource in package.get("resources", []))
resource_df

vornamen-von-neugeborenen
Vornamen von Nulljährigen München
Vornamen des jeweiligen Jahrgangs der Münchner Hauptwohnsitzbevölkerung (Nulljährige) am 31.12. des jeweiligen Jahres (unabhängig vom Geburtsort).


,id,name,description,format,mimetype,url,datastore_active,created,last_modified
0,6388c83a-266d-437c-824a-7bbcb7ceec63,Vornamen 2025,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/6388c83a-266d-437c-824a-7bbcb7ceec63/download/vorname...,True,2026-04-13T09:45:22.337339,2026-04-13T09:45:22.255869
1,9f393a00-aa64-4733-8c8c-80eabc18f95a,Vornamen 2024,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/9f393a00-aa64-4733-8c8c-80eabc18f95a/download/vorname...,True,2025-08-07T05:49:46.213043,2025-08-07T05:49:46.043132
2,02ab322e-d33e-4447-a9ab-23df63dfa7e1,Vornamen 2023,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/02ab322e-d33e-4447-a9ab-23df63dfa7e1/download/vorname...,True,2024-02-12T08:54:48.386029,2025-08-07T07:01:15.365302
3,5f11d3a0-4779-4f64-b113-b59326e6d839,Vornamen 2022,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/5f11d3a0-4779-4f64-b113-b59326e6d839/download/open_da...,True,2023-01-26T16:37:07.125773,2025-08-07T07:01:42.731521
4,dc6170db-f7f4-4bdc-b790-13df55f0cf64,Vornamen 2021,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/dc6170db-f7f4-4bdc-b790-13df55f0cf64/download/vorname...,True,2023-01-25T09:24:38.524115,2025-08-07T07:02:43.823891
5,48a4c5fd-f5f2-4c0c-bcb3-222fd9ebda67,Vornamen 2020,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/48a4c5fd-f5f2-4c0c-bcb3-222fd9ebda67/download/vorname...,True,2023-01-25T09:24:00.347985,2025-08-07T07:03:05.791416
6,523d5860-25eb-4bea-96e3-193d1dacfb8f,Vornamen 2019,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/523d5860-25eb-4bea-96e3-193d1dacfb8f/download/vorname...,True,2023-01-25T09:23:25.150115,2023-01-25T09:23:25.102467
7,23fdd53e-485a-46bd-abb6-d7a51b2ebb18,Vornamen 2018,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/23fdd53e-485a-46bd-abb6-d7a51b2ebb18/download/vorname...,True,2023-01-25T09:22:23.137131,2023-01-25T09:22:23.087525
8,2c70d289-8b1a-4071-9739-866c5e532b77,Vornamen 2017,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/2c70d289-8b1a-4071-9739-866c5e532b77/download/vorname...,True,2023-01-25T09:21:49.782968,2023-01-25T09:21:49.741012
9,6655ec3c-251a-4a5e-9a57-d60c4ad59ca3,Vornamen 2016,,CSV,text/csv,https://opendata.muenchen.de/dataset/99ad40ec-9d7b-4a2e-87eb-9bac783fb57a/resource/6655ec3c-251a-4a5e-9a57-d60c4ad59ca3/download/vorname...,True,2023-01-25T09:21:20.653552,2023-01-25T09:21:20.612349


## Sample a resource

Some CKAN resources are available through the datastore API. For datastore-backed resources, use `datastore_search` and pass filters to CKAN. For ordinary CSV, XML, and HTML resources, fetch the file URL and filter client-side after loading into pandas.

This distinction matters for tool calling:

- `datastore_active=True`: the model can call `fetch_resource(resource_id, filters={...})` and the backend can push filters to CKAN.
- static file URL: the model can call `fetch_resource(resource_id)`, then the backend samples/parses the file and applies filters locally.

In [32]:
class SimpleHTMLTableParser(HTMLParser):
    """Small stdlib table parser for simple HTML tables."""

    def __init__(self) -> None:
        super().__init__()
        self.tables: list[list[list[str]]] = []
        self._current_table: list[list[str]] | None = None
        self._current_row: list[str] | None = None
        self._current_cell: list[str] | None = None
        self._in_cell = False

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag == "table":
            self._current_table = []
        elif tag == "tr" and self._current_table is not None:
            self._current_row = []
        elif tag in {"td", "th"} and self._current_row is not None:
            self._current_cell = []
            self._in_cell = True

    def handle_data(self, data: str) -> None:
        if self._in_cell and self._current_cell is not None:
            text = data.strip()
            if text:
                self._current_cell.append(text)

    def handle_endtag(self, tag: str) -> None:
        if tag in {"td", "th"} and self._current_row is not None and self._current_cell is not None:
            self._current_row.append(" ".join(self._current_cell).strip())
            self._current_cell = None
            self._in_cell = False
        elif tag == "tr" and self._current_table is not None and self._current_row is not None:
            if self._current_row:
                self._current_table.append(self._current_row)
            self._current_row = None
        elif tag == "table" and self._current_table is not None:
            if self._current_table:
                self.tables.append(self._current_table)
            self._current_table = None


def fetch_text(url: str) -> str:
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=30) as response:
        content_type = response.headers.get_content_charset() or "utf-8"
        return response.read().decode(content_type, errors="replace")


def apply_client_filters(df: pd.DataFrame, filters: dict[str, Any] | None = None) -> pd.DataFrame:
    """Apply equality or membership filters to a dataframe."""
    if not filters:
        return df

    filtered = df.copy()
    for column, expected in filters.items():
        if column not in filtered.columns:
            raise KeyError(f"Filter column not found: {column}. Available columns: {list(filtered.columns)}")
        if isinstance(expected, list):
            filtered = filtered[filtered[column].isin(expected)]
        else:
            filtered = filtered[filtered[column] == expected]
    return filtered


def sample_csv_resource(resource: dict[str, Any], limit: int = 10, filters: dict[str, Any] | None = None) -> pd.DataFrame:
    """Sample CSV/TXT resources. Filtering is client-side for plain files."""
    df = pd.read_csv(resource["url"], sep=None, engine="python", nrows=None if filters else limit)
    return apply_client_filters(df, filters).head(limit)


def _flatten_xml_element(element: ET.Element) -> dict[str, Any]:
    row: dict[str, Any] = dict(element.attrib)
    for child in list(element):
        key = child.tag.split("}")[-1]
        value = (child.text or "").strip()
        if list(child):
            nested = _flatten_xml_element(child)
            for nested_key, nested_value in nested.items():
                row[f"{key}.{nested_key}"] = nested_value
        elif value:
            row[key] = value
    return row


def sample_xml_resource(resource: dict[str, Any], limit: int = 10, filters: dict[str, Any] | None = None) -> pd.DataFrame:
    """Sample XML by flattening repeated leaf-like elements into rows."""
    text = fetch_text(resource["url"])
    root = ET.fromstring(text)

    candidates = []
    for element in root.iter():
        children = list(element)
        if children:
            row = _flatten_xml_element(element)
            if row:
                candidates.append(row)

    if not candidates:
        candidates = [_flatten_xml_element(root)]

    df = pd.DataFrame(candidates)
    return apply_client_filters(df, filters).head(limit)


def sample_html_resource(resource: dict[str, Any], limit: int = 10, filters: dict[str, Any] | None = None, table_index: int = 0) -> pd.DataFrame:
    """Sample the first simple HTML table using only the Python standard library."""
    parser = SimpleHTMLTableParser()
    parser.feed(fetch_text(resource["url"]))

    if not parser.tables:
        raise ValueError("No HTML tables found in resource.")

    table = parser.tables[table_index]
    header, rows = table[0], table[1:]
    width = len(header)
    normalized_rows = [row[:width] + [None] * max(0, width - len(row)) for row in rows]
    df = pd.DataFrame(normalized_rows, columns=header)
    return apply_client_filters(df, filters).head(limit)


def sample_datastore_resource(resource: dict[str, Any], limit: int = 10, filters: dict[str, Any] | None = None) -> pd.DataFrame:
    """Sample datastore-backed resources with server-side CKAN filters."""
    params: dict[str, Any] = {"resource_id": resource["id"], "limit": limit}
    if filters:
        params["filters"] = json.dumps(filters, ensure_ascii=False)
    result = ckan_action("datastore_search", params)
    return pd.DataFrame(result.get("records", []))


def sample_resource(resource: dict[str, Any], limit: int = 10, filters: dict[str, Any] | None = None) -> pd.DataFrame:
    """Return a sample dataframe from datastore, CSV/TXT, XML, or HTML resources."""
    if resource.get("datastore_active"):
        return sample_datastore_resource(resource, limit=limit, filters=filters)

    url = resource.get("url")
    fmt = (resource.get("format") or "").lower()
    if not url:
        raise ValueError("Resource has no URL.")
    if fmt in {"csv", "txt"}:
        return sample_csv_resource(resource, limit=limit, filters=filters)
    if fmt in {"xml", "rdf", "rss", "atom"} or url.lower().endswith(".xml"):
        return sample_xml_resource(resource, limit=limit, filters=filters)
    if fmt in {"html", "htm"} or url.lower().endswith((".html", ".htm")):
        return sample_html_resource(resource, limit=limit, filters=filters)

    raise ValueError(f"No sampler implemented for resource format={resource.get('format')} url={url}")


candidate_resources = [
    resource for resource in package.get("resources", [])
    if resource.get("datastore_active") or (resource.get("format") or "").lower() in {"csv", "txt", "xml", "rdf", "rss", "atom", "html", "htm"}
]

print(f"sampleable resources: {len(candidate_resources)}")
selected_resource = candidate_resources[0]
print(selected_resource["id"], selected_resource.get("name"), selected_resource.get("format"), selected_resource.get("datastore_active"))

sample_df = sample_resource(selected_resource, limit=20)
sample_df

sampleable resources: 13
6388c83a-266d-437c-824a-7bbcb7ceec63 Vornamen 2025 CSV True


,_id,vorname,anzahl,geschlecht
0,1,Felix,89,m
1,2,Anton,87,m
2,3,Emma,81,w
3,4,Emil,80,m
4,5,Clara,79,w
5,6,Leon,76,m
6,7,Emilia,74,w
7,8,Leo,74,m
8,9,Noah,73,m
9,10,Theo,73,m


For datastore-backed resources, inspect fields and try simple filters. These filters are the kind of structured arguments the function-calling model should learn to emit.

In [33]:
if selected_resource.get("datastore_active"):
    datastore_result = ckan_action("datastore_search", {"resource_id": selected_resource["id"], "limit": 5})
    print("fields:")
    display(pd.DataFrame(datastore_result.get("fields", [])))
    print("records:")
    display(pd.DataFrame(datastore_result.get("records", [])))
else:
    print("Selected resource is not datastore-backed. Columns from pandas sample:")
    display(pd.DataFrame({"column": sample_df.columns, "dtype": [str(dtype) for dtype in sample_df.dtypes]}))

fields:


,id,type
0,_id,int
1,vorname,text
2,anzahl,text
3,geschlecht,text


records:


,_id,vorname,anzahl,geschlecht
0,1,Felix,89,m
1,2,Anton,87,m
2,3,Emma,81,w
3,4,Emil,80,m
4,5,Clara,79,w


## Build a compact catalog snapshot

This snapshot is source material for training-data generation. Keep it small at first, then increase `MAX_PACKAGES` once the shape is stable.

In [34]:
MAX_PACKAGES = 25
PAGE_SIZE = 10

catalog_packages = []
for start in range(0, MAX_PACKAGES, PAGE_SIZE):
    result = search_packages("", rows=min(PAGE_SIZE, MAX_PACKAGES - start), start=start, sleep_s=0.1)
    for package in result["results"]:
        catalog_packages.append(compact_package(package))

len(catalog_packages)

25

In [35]:
catalog_rows = []
for package in catalog_packages:
    for resource in package["resources"]:
        catalog_rows.append({
            "package_name": package["name"],
            "package_title": package["title"],
            "resource_id": resource["id"],
            "resource_name": resource["name"],
            "format": resource["format"],
            "datastore_active": resource["datastore_active"],
            "tags": package["tags"],
        })

catalog_df = pd.DataFrame(catalog_rows)
catalog_df.head(30)

,package_name,package_title,resource_id,resource_name,format,datastore_active,tags
0,solarpotenzial_globalstrahlung_p_02,Solarpotenzialanalyse 2018 der Landeshauptstadt München,160ad084-a469-4718-9d82-68ee8ccebdf9,Metadaten (XML),XML,False,"[Digitaler Zwilling München, Landeshauptstadt München, Planungsunterlagen Kataster]"
1,solarpotenzial_globalstrahlung_p_02,Solarpotenzialanalyse 2018 der Landeshauptstadt München,48b0fe46-4592-43c2-b35c-fdbec8744a26,WMS (GetCapabilities),WMS,False,"[Digitaler Zwilling München, Landeshauptstadt München, Planungsunterlagen Kataster]"
2,solarpotenzial_globalstrahlung_p_02,Solarpotenzialanalyse 2018 der Landeshauptstadt München,a1e527c2-9a8b-4fc3-982a-7889683106f6,Fachportal Energie,HTML,False,"[Digitaler Zwilling München, Landeshauptstadt München, Planungsunterlagen Kataster]"
3,solarpotenzial_globalstrahlung_p_02,Solarpotenzialanalyse 2018 der Landeshauptstadt München,f0242692-a06d-4908-a12d-fb57211c931a,Open Geodata Portal des Geoportals,HTML,False,"[Digitaler Zwilling München, Landeshauptstadt München, Planungsunterlagen Kataster]"
4,enp_baublock_grundwasserpotential_einteilig_25832,Grundwasserpotential,04a6c65d-3b01-4623-9e73-ad6babed6d5c,Metadaten (XML),XML,False,"[Digitaler Zwilling München, Landeshauptstadt München, Planungsunterlagen Kataster]"
5,enp_baublock_grundwasserpotential_einteilig_25832,Grundwasserpotential,964baca6-844f-4132-9526-2ce216674988,WMS (GetCapabilities),WMS,False,"[Digitaler Zwilling München, Landeshauptstadt München, Planungsunterlagen Kataster]"
6,enp_baublock_grundwasserpotential_einteilig_25832,Grundwasserpotential,4d0e4d13-14e7-4dad-ab4a-388ea587d676,Fachportal Energie,HTML,False,"[Digitaler Zwilling München, Landeshauptstadt München, Planungsunterlagen Kataster]"
7,enp_baublock_grundwasserpotential_einteilig_25832,Grundwasserpotential,edde8548-855f-42bb-8822-f3958407e01d,Open Geodata Portal des Geoportals,HTML,False,"[Digitaler Zwilling München, Landeshauptstadt München, Planungsunterlagen Kataster]"
8,abfall_sonstiges_opendata,Sonstige Abfallentsorgungsanlagen (Opendata),ed5cf769-d6d1-4bfc-8073-9e742b10223e,Metadaten (XML),XML,False,"[Digitaler Zwilling München, Opendata, Umweltschutz, infoMapAccessService]"
9,abfall_sonstiges_opendata,Sonstige Abfallentsorgungsanlagen (Opendata),dad86986-e65f-42ac-9331-17171935899b,WMS (GetCapabilities),WMS,False,"[Digitaler Zwilling München, Opendata, Umweltschutz, infoMapAccessService]"


Add lightweight schema and row samples for resources that can be sampled. Failures are stored so the generator can skip unsupported resources later.

In [36]:
def enrich_resource_sample(resource: dict[str, Any], sample_limit: int = 5) -> dict[str, Any]:
    enriched = dict(resource)
    try:
        sample = sample_resource(resource, limit=sample_limit)
        enriched["schema"] = [{"name": str(column), "dtype": str(dtype)} for column, dtype in sample.dtypes.items()]
        enriched["sample_rows"] = sample.where(pd.notna(sample), None).to_dict(orient="records")
        enriched["sample_error"] = None
    except Exception as exc:
        enriched["schema"] = []
        enriched["sample_rows"] = []
        enriched["sample_error"] = str(exc)
    return enriched


enriched_catalog = []
for package in catalog_packages:
    enriched_package = dict(package)
    enriched_package["resources"] = [
        enrich_resource_sample(resource)
        for resource in package["resources"]
    ]
    enriched_catalog.append(enriched_package)
    time.sleep(0.1)

enriched_catalog[0]

{'id': '1ae5ba90-a8a3-4c95-a310-83e9e077919f',
 'name': 'solarpotenzial_globalstrahlung_p_02',
 'title': 'Solarpotenzialanalyse 2018 der Landeshauptstadt München',
 'notes': 'Basierend auf einem 3D-Modell, stellt die Solarpotenzialanalyse auf Grundlage von Ausrichtung, Neigung und Verschattung die einfallende Globalstrahlung in kWh/m2a auf alle Dachflächen dar. Die Analyse berücksichtigt Formparameter der Dachfläche und Denkmalschutzbelange. Die Solarpotenzialanalyse dient zur Erstinformation über vorhandene Solarpotenziale für interessierte Bürgerinnen und Bürger.\n\nDas Referat für Stadtplanung und Bauordnung stellt die Daten mit der erforderlichen Sorgfalt bereit. Eine Gewähr für die Richtigkeit, Vollzähligkeit, Maßhaltigkeit und Genauigkeit der überlassenen Daten wird nicht übernommen. Insoweit besteht auch keine Haftung für unrichtige Angaben, Übertragungsfehler, Folgeschäden oder sonstige Schäden jeglicher Art.',
 'tags': ['Digitaler Zwilling München',
  'Landeshauptstadt München

Write the compact snapshot as JSONL. This file is the first input to future scripts such as `generate_tool_examples.py` and `generate_openui_examples.py`.

In [37]:
catalog_path = RAW_DATA_DIR / "munich_catalog_sample.jsonl"

with catalog_path.open("w", encoding="utf-8") as file:
    for package in enriched_catalog:
        file.write(json.dumps(package, ensure_ascii=False) + "\n")

catalog_path, catalog_path.stat().st_size

(PosixPath('../training/data/raw/munich_catalog_sample.jsonl'), 1420891)

## Build basic dataset statistics JSONL

Create one JSONL record per CKAN dataset/package. Each record keeps the dataset description and resource metadata, then inspects sampleable resources for row counts, columns, missing values, example values, and numeric summary statistics. Datastore-backed resources expose exact `entry_count` through CKAN; static files are profiled from a bounded sample so the notebook remains safe to run against the whole portal.

In [ ]:
DATASET_PROFILE_MAX_PACKAGES: int | None = None  # Set to a small number while iterating, for example 25.
DATASET_PROFILE_PAGE_SIZE = 25
DATASET_PROFILE_SAMPLE_LIMIT = 1_000
DATASET_PROFILE_SLEEP_S = 0.05


def json_safe(value: Any) -> Any:
    """Convert pandas/numpy scalars and missing values into JSON-safe Python values."""
    if value is None:
        return None
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        missing = False
    if isinstance(missing, bool) and missing:
        return None
    if hasattr(value, "item"):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def unique_examples(series: pd.Series, limit: int = 8) -> list[Any]:
    values = []
    for value in series.dropna():
        safe_value = json_safe(value)
        if safe_value not in values:
            values.append(safe_value)
        if len(values) >= limit:
            break
    return values


def numeric_summary(series: pd.Series) -> dict[str, Any] | None:
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if numeric.empty:
        return None
    return {
        "count": int(numeric.count()),
        "mean": json_safe(numeric.mean()),
        "min": json_safe(numeric.min()),
        "max": json_safe(numeric.max()),
        "median": json_safe(numeric.median()),
    }


def summarize_dataframe(df: pd.DataFrame, entry_count: int | None = None, entry_count_source: str = "sample") -> dict[str, Any]:
    columns = []
    for column in df.columns:
        series = df[column]
        summary = {
            "name": str(column),
            "dtype": str(series.dtype),
            "non_null_count": int(series.notna().sum()),
            "null_count": int(series.isna().sum()),
            "example_values": unique_examples(series),
        }
        stats = numeric_summary(series)
        if stats is not None:
            summary["numeric"] = stats
        columns.append(summary)

    return {
        "entry_count": entry_count,
        "entry_count_source": entry_count_source,
        "sampled_rows": int(len(df)),
        "column_count": int(len(df.columns)),
        "columns": columns,
    }


def summarize_datastore_resource(resource: dict[str, Any], sample_limit: int = DATASET_PROFILE_SAMPLE_LIMIT) -> dict[str, Any]:
    result = ckan_action("datastore_search", {"resource_id": resource["id"], "limit": sample_limit})
    records = result.get("records", [])
    df = pd.DataFrame(records)
    summary = summarize_dataframe(df, entry_count=result.get("total"), entry_count_source="ckan_datastore_total")
    if not summary["columns"] and result.get("fields"):
        summary["column_count"] = len(result["fields"])
        summary["columns"] = [
            {"name": field.get("id") or field.get("name"), "dtype": field.get("type"), "non_null_count": None, "null_count": None, "example_values": []}
            for field in result["fields"]
        ]
    return summary


def summarize_static_resource(resource: dict[str, Any], sample_limit: int = DATASET_PROFILE_SAMPLE_LIMIT) -> dict[str, Any]:
    df = sample_resource(resource, limit=sample_limit)
    entry_count = len(df) if len(df) < sample_limit else None
    entry_count_source = "complete_sample" if entry_count is not None else "bounded_sample"
    return summarize_dataframe(df, entry_count=entry_count, entry_count_source=entry_count_source)


def profile_resource_statistics(resource: dict[str, Any], sample_limit: int = DATASET_PROFILE_SAMPLE_LIMIT) -> dict[str, Any]:
    profile = compact_resource(resource)
    try:
        if resource.get("datastore_active"):
            profile["table"] = summarize_datastore_resource(resource, sample_limit=sample_limit)
        else:
            profile["table"] = summarize_static_resource(resource, sample_limit=sample_limit)
        profile["profile_error"] = None
    except Exception as exc:
        profile["table"] = None
        profile["profile_error"] = str(exc)
    return profile


def profile_package_statistics(package: dict[str, Any], sample_limit: int = DATASET_PROFILE_SAMPLE_LIMIT) -> dict[str, Any]:
    resources = [profile_resource_statistics(resource, sample_limit=sample_limit) for resource in package.get("resources", [])]
    column_names = sorted({
        column["name"]
        for resource in resources
        for column in ((resource.get("table") or {}).get("columns") or [])
        if column.get("name")
    })
    return {
        "id": package.get("id"),
        "name": package.get("name"),
        "title": package.get("title"),
        "description": package.get("notes"),
        "tags": [tag.get("name") for tag in package.get("tags", [])],
        "groups": [group.get("name") for group in package.get("groups", [])],
        "organization": (package.get("organization") or {}).get("title"),
        "metadata_created": package.get("metadata_created"),
        "metadata_modified": package.get("metadata_modified"),
        "resource_count": len(package.get("resources", [])),
        "formats": sorted({resource.get("format") for resource in package.get("resources", []) if resource.get("format")}),
        "column_count": len(column_names),
        "columns": column_names,
        "resources": resources,
    }


def iter_packages(max_packages: int | None = DATASET_PROFILE_MAX_PACKAGES, page_size: int = DATASET_PROFILE_PAGE_SIZE):
    start = 0
    emitted = 0
    while True:
        rows = page_size if max_packages is None else min(page_size, max_packages - emitted)
        if rows <= 0:
            break
        result = search_packages("", rows=rows, start=start, sleep_s=DATASET_PROFILE_SLEEP_S)
        packages = result.get("results", [])
        if not packages:
            break
        for package in packages:
            yield package
            emitted += 1
            if max_packages is not None and emitted >= max_packages:
                return
        start += len(packages)
        if start >= result.get("count", 0):
            break


Run the exporter. While developing, set `DATASET_PROFILE_MAX_PACKAGES` above to a small number. With the default `None`, it walks the complete CKAN package list and writes `munich_dataset_basic_statistics.jsonl`.

In [ ]:
dataset_statistics_path = RAW_DATA_DIR / "munich_dataset_basic_statistics.jsonl"

written = 0
with dataset_statistics_path.open("w", encoding="utf-8") as file:
    for package in iter_packages():
        profile = profile_package_statistics(package)
        file.write(json.dumps(profile, ensure_ascii=False) + "\n")
        written += 1
        if written % 25 == 0:
            print(f"profiled {written:,} datasets")

dataset_statistics_path, written, dataset_statistics_path.stat().st_size

## Next exploration questions

- Which resource formats are most common?
- Which resources support `datastore_search`?
- Which datasets have clean tabular schemas that are useful for a demo?
- Which German and English user questions map cleanly to high-confidence dataset search queries?
- Which filters are available from actual resource columns, for example year, district, category, or date?